In [ ]:
# 従来型のWebスクレイピング

import requests
from bs4 import BeautifulSoup

# ニュース一覧ページのURL
url = "https://www.kiramex.com/news/"

# ページのHTMLを取得
response = requests.get(url)
response.encoding = response.apparent_encoding  # 文字コードを自動判別して設定

# HTMLをBeautifulSoupでパース
soup = BeautifulSoup(response.text, "html.parser")

# ニュース一覧の情報を抽出
news_list = soup.find_all("li", class_="news-block")

# ニュース情報の取得
for news in news_list:
    # ニュースの日付を取得
    date = (
        news.find("p", class_="date").get_text(strip=True)
        if news.find("p", class_="date")
        else "N/A"
    )

    # ニュースのタイトルを取得
    title = news.find("a", class_="title").get_text(strip=True)

    # ニュースのリンクを取得
    link = news.find("a", class_="title")["href"]

    print(f"Date: {date}")
    print(f"Title: {title}")
    print(f"Link: {link}")
    print("-" * 40)

SSLError: HTTPSConnectionPool(host='www.kiramex.com', port=443): Max retries exceeded with url: /news/ (Caused by SSLError(SSLError(1, '[SSL: SSLV3_ALERT_HANDSHAKE_FAILURE] ssl/tls alert handshake failure (_ssl.c:1010)')))

In [3]:
# 言語モデルを活用したスクレイピング

# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from openai import OpenAI

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ["API_KEY"])

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [ ]:
# ニュース一覧ページのURL

# 20260416 一時的に別のURLに変更: url = "https://www.kiramex.com/news/"
url = "https://www.lycorp.co.jp/ja/news/"

# ページのHTMLを取得
response = requests.get(url)
response.encoding = response.apparent_encoding  # 文字コードを自動判別して設定

# HTMLをBeautifulSoupでパースし、body部分を取り出す
soup = BeautifulSoup(response.text, "html.parser")
body_html = str(soup.body)  # body部分のHTMLを文字列として取得
print(body_html)  # 結果を表示して確認

<body>
<!-- Google Tag Manager (noscript) -->
<noscript><iframe height="0" src="https://www.googletagmanager.com/ns.html?id=GTM-NJPSPSD5" style="display:none;visibility:hidden" width="0"></iframe></noscript>
<!-- End Google Tag Manager (noscript) -->
<script src="/ja/ja_header2.js"></script>
<div class="global-container">
<nav class="breadcrumbs">
<ul>
<li><a href="/ja/"><img alt="トップページ" height="14" src="/assets/images/icon_home.svg" width="14"/></a></li>
<li>ニュース</li>
</ul>
</nav>
<div class="page-heading-lv2">
<div class="inner">
<h1>ニュース</h1>
<p>News</p>
</div>
<picture>
<source media="(max-width: 767px)" srcset="/assets/images/news/sp/LY_siteKV_news.jpg"/>
<img alt="" src="/assets/images/news/LY_siteKV_news.jpg"/>
</picture>
</div>
<div class="body-container no-top-padding">
<div class="body-inner search-target">
<div class="c-search-header" id="news-search">
<h2 id="press-release">プレスリリース</h2>
<button class="c-search-open" type="button">絞り込み検索</button>
</div>
<ul class="c-col-set

In [6]:
# LLMにニュース一覧を抽出させるプロンプトを作成

# 「ニュース一覧」から「プレスリリース一覧」に変更

prompt = f"""
以下のHTMLから最新のプレスリリース一覧を抽出し、「日付、タイトル、リンク」の形式で一覧を出力してください。一覧以外は出力しないでください。

# 出力様式：
Date: 日付
Title: タイトル
Link: リンク
--------------------

#HTML:
{body_html[:5000]}
"""

# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": prompt},
    ],
    max_tokens=500,
    temperature=0.3,
)

# LLMからの回答を表示
print(response.choices[0].message.content.strip())

```
Date: 2026年4月14日
Title: Yahoo! JAPAN ID、安全性・利便性向上を目的としてログイン方法を「パスキー」に一本化へ
Link: https://www.lycorp.co.jp/ja/news/release/020333/
--------------------
Date: 2026年4月13日
Title: 「LINEミニアプリ」、デジタルコンテンツ課金機能を本格提供
Link: https://www.lycorp.co.jp/ja/news/release/020192/
--------------------
Date: 2026年4月10日
Title: LINEヤフー、LINEをはじめ9アプリでStarlink衛星通信に初めて対応  山間部や海上、災害時などキャリア通信が圏外でもアプリの利用が可能に
Link: https://www.lycorp.co.jp/ja/news/release/020253/
--------------------
Date: 2026年4月2日
Title: LINEヤフー、ディスプレイ広告プラットフォームを統合  新たに「LINEヤフー広告 ディスプレイ広告」の提供を開始
Link: https://www.lycorp.co.jp/ja/news/release/020307/
--------------------
Date: 2026年4月1日
Title: 「LINE公式アカウント」の認証バッジを刷新、 国内の認証済アカウントは緑色のチェックマークに統一
Link: https://www.lycorp.co.jp/ja/news/release/020255/
--------------------
Date: 2026年4月1日
Title: LINEヤフー、新たな事業拠点として赤坂オフィスを開設
Link: https://www.lycorp.co.jp/ja/news/release/020276/
--------------------
Date: 2026年3月30日
Title: LINE証券、CFD取引サービス「LINE CFD」の提供を再開  多くのユーザーの要望を受け、利便性の高い取引環境を提供
Link: https://www.ly